In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"  
# os.environ["CUDA_VISIBLE_DEVICES"] = "2,3"  
from torch import nn


import numpy as np                                                                  
import torch                                          

                              
from transformers import AutoModelForCausalLM, AutoTokenizer  
from tqdm import tqdm
import matplotlib.pyplot as plt  
import torch.nn.functional as F
import gc
import re
import copy

import sys
sys.path.append('..')
import JCBScope_utils
import JacobianScopes

# Move to GPU with optimal dtype
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# device = "cpu"


/home/jl3499/conda/LLM1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/jl3499/conda/LLM1/lib/python3.12/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
# Load the tokenizer and model

# model_name = "meta-llama/Llama-3.2-1B"
model_name = "meta-llama/Llama-3.2-3B"
# model_name = "meta-llama/Llama-3.1-8B"
model_name_short = model_name.split("/")[-1]
if device == "cpu":
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)
    model = model.to(device)
else:
    tokenizer = AutoTokenizer.from_pretrained(model_name, device_map="auto")
    model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")
    
embedding_layer = model.get_input_embeddings()
embed_device = embedding_layer.weight.device    

Loading checkpoint shards: 100%|██████████| 2/2 [00:03<00:00,  1.54s/it]


In [3]:
# mode = 'Temperature'
# mode = 'Semantic'
mode = 'gradient_x_input'
# mode = 'IG'
presence_list = [0.2,0.4,0.6,0.8,1]




In [4]:
top_k_fraction = 0.05
# top_k_fraction = 0.1
# top_k_fraction = 0.2

In [5]:
front_pad = 0
back_pad = 0

front_strip = 0
# random_ablation = True
random_ablation = False
# num_prompts = 1
num_prompts = 300


# Get special tokens if available
bos_token_id = tokenizer.bos_token_id if tokenizer.bos_token_id is not None else tokenizer.cls_token_id
eos_token_id = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else tokenizer.sep_token_id


In [6]:
if (mode == 'Semantic') or (mode == 'IG'):
    unnormalized_logits = True
else:
    unnormalized_logits = False
    

def change_in_max_log_p_with_ablation(string, top_k_fraction=0.1, verbose=True, random_ablation = False):
    """Optimized: single model load, retain_graph=False, free logits before ablation."""
    input_ids_list = tokenizer(string, add_special_tokens=False)["input_ids"]
    if eos_token_id is not None:
        input_ids_list += [eos_token_id] * back_pad

    input_ids = torch.tensor([input_ids_list], dtype=torch.long).to(embed_device)
    attention_mask = torch.ones_like(input_ids, device=embed_device)
    seq_len = input_ids.size(1)

    decoded_tokens = tokenizer.batch_decode([[tid] for tid in input_ids[0].tolist()], skip_special_tokens=True)

    # def get_token_indices_for_substring(tokens, substring):
    #     full = "".join(tokens)
    #     exclude_indices = set()
    #     start = 0
    #     while True:
    #         pos = full.find(substring, start)
    #         if pos == -1:
    #             break
    #         end = pos + len(substring)
    #         cumlen = 0
    #         for i, t in enumerate(tokens):
    #             tok_end = cumlen + len(t)
    #             if tok_end > pos and cumlen < end:
    #                 exclude_indices.add(i)
    #             cumlen = tok_end
    #         start = pos + 1
    #     return exclude_indices

    grad_idx = [idx for idx in range(front_pad, len(decoded_tokens), 1)][front_strip:]

    tick_label_text = [decoded_tokens[idx] for idx in grad_idx]

    d_model = embedding_layer.embedding_dim
    residual = nn.Parameter(torch.zeros(len(grad_idx), d_model, device=embed_device))
    presence = torch.ones(len(decoded_tokens), 1, device=embed_device)
    # Use global model (no reload - avoids OOM from loading 2x per call)
    model.eval()
    forward_pass = JCBScope_utils.customize_forward_pass(
        model, residual, presence, input_ids, grad_idx, attention_mask
    )
    hidden_norm_as_loss = (mode == 'Temperature')
    loss_position = seq_len - 2

    if verbose:
        print(f"Computing {mode} Scope...")
    n_tokens = len(grad_idx)
    n_top = max(1, int(n_tokens * top_k_fraction))
    if random_ablation:
        with torch.no_grad():
            _, logits_orig = forward_pass(
                loss_position=loss_position,
                hidden_norm_as_loss=hidden_norm_as_loss,
                unnormalized_logits=True,
                tie_input_output_embed=False,
            )
        grad_vals = None
        ablated_indices = np.random.choice(n_tokens, size=min(n_top, n_tokens), replace=False)
    else:
        if mode == "IG":
            grad_list = []
            for alpha in presence_list:
                loss, logits_orig = forward_pass(
                    loss_position=loss_position,
                    hidden_norm_as_loss=hidden_norm_as_loss,
                    unnormalized_logits=False,
                    tie_input_output_embed=False,
                    alpha=alpha,
                )
                g = torch.autograd.grad(loss, residual, retain_graph=False)[0]
                grad_list.append(g.detach().clone())
                del loss
            grads = torch.stack(grad_list).mean(dim=0)
            del grad_list
            with torch.no_grad():
                token_embeds = JCBScope_utils.embedding_lookup(input_ids[0, grad_idx], embedding_layer)
            grad_vals = (grads * token_embeds.to(grads.device)).norm(dim=-1).squeeze().cpu().numpy()
            del grads
        elif mode == "Temperature":
            grad_vals, logits_orig = JacobianScopes.temperature_scope_scores(forward_pass, residual, loss_position)

        elif mode == "Semantic":
            grad_vals, logits_orig = JacobianScopes.semantic_scope_scores(forward_pass, residual, loss_position)

        elif mode == "gradient_x_input":
            grad_vals, logits_orig = JacobianScopes.gradient_x_input_scores(
                forward_pass, residual, loss_position, embedding_layer, input_ids, grad_idx
            )
        else:
            raise ValueError(f"Unknown mode: {mode!r}")
        if grad_vals.ndim > 1:
            grad_vals = grad_vals.squeeze()
        ablated_indices = grad_vals.argsort()[::-1][:n_top]

    max_idx_orig = logits_orig[loss_position].argmax().item()
    max_log_prob_orig = torch.log_softmax(logits_orig[loss_position], dim=-1)[max_idx_orig].item()
    logits_orig_cpu = logits_orig.detach().cpu()
    del logits_orig
    torch.cuda.empty_cache()
    presence_ablated = presence.clone()
    presence_ablated[[grad_idx[i] for i in ablated_indices], 0] = 0.0
    del forward_pass

    forward_pass_ablated = JCBScope_utils.customize_forward_pass(
        model, residual, presence_ablated, input_ids, grad_idx, attention_mask
    )
    with torch.no_grad():
        _, logits_ablated = forward_pass_ablated(
            loss_position=loss_position,
            hidden_norm_as_loss=hidden_norm_as_loss,
            unnormalized_logits=unnormalized_logits,
            tie_input_output_embed=False,
        )

    max_log_prob_ablated = torch.log_softmax(logits_ablated[loss_position], dim=-1)[max_idx_orig].item()
    delta_log_prob = max_log_prob_ablated - max_log_prob_orig
    del logits_ablated, forward_pass_ablated
    torch.cuda.empty_cache()

    true_token_id = input_ids[0, loss_position + 1].item()
    predicted_token_id = max_idx_orig
    true_token_str = tokenizer.decode([true_token_id])
    predicted_token_str = tokenizer.decode([predicted_token_id])
    match = (true_token_id == predicted_token_id)

    if verbose:
        # print(string)
        dropped_words = []
        for i in ablated_indices:
            tok = input_ids[0, grad_idx[int(i)]].item()
            word = tokenizer.decode([tok]).strip()
            dropped_words.append(word)
        # print(f"Words dropped: {dropped_words}")
        # print(f"True token: {true_token_str!r}")
        # print(f"Predicted token: {predicted_token_str!r}")
        # print(f"Match: {match}")

    return delta_log_prob, grad_vals, ablated_indices, tick_label_text, logits_orig_cpu, true_token_str, predicted_token_str

In [ ]:
import json
from pathlib import Path

# Load prompts from JSON
prompts_path = Path("../data/lambada_prompts.json")
with open(prompts_path, "r", encoding="utf-8") as f:
    all_prompts_data = json.load(f)

# Handle both list-of-dicts and {prompts: [...]} formats
if isinstance(all_prompts_data, dict) and "prompts" in all_prompts_data:
    prompts_list = all_prompts_data["prompts"]
else:
    prompts_list = all_prompts_data


prompts_to_process = prompts_list[:num_prompts]

# Label: model name + scope type
label = f"{model_name_short}__{mode}_lambada_top{top_k_fraction}"
if random_ablation:
    label += "_random_drop"
results = []
for i, item in enumerate(tqdm(prompts_to_process, desc="Processing prompts")):
    # print (f"processing {i+1} of {num_prompts} prompts")
    prompt = item["text"] if isinstance(item, dict) else item

    delta_log_prob, grad_vals, ablated_indices, tick_label_text, logits_orig, true_token, predicted_token = change_in_max_log_p_with_ablation(string=prompt, top_k_fraction=top_k_fraction, verbose=True, random_ablation=random_ablation)
    # Keep last successful result for visualization cells
    _last_grad_vals, _last_ablated, _last_tick = grad_vals, ablated_indices, tick_label_text
    _last_logits = logits_orig.detach().cpu() if hasattr(logits_orig, "cpu") else logits_orig
    del logits_orig
    entry = {
        "delta_log_prob": delta_log_prob,
        "prompt": prompt,
        "true_token": true_token,
        "predicted_token": predicted_token,
        "index": i,
    }
    if isinstance(item, dict):
        entry.update({k: v for k, v in item.items() if k != "prompt"})
    results.append(entry)




Processing prompts:   0%|          | 0/300 [00:00<?, ?it/s]

Computing gradient_x_input Scope...


/home/jl3499/conda/LLM1/lib/python3.12/site-packages/torch/autograd/graph.py:769: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /home/conda/feedstock_root/build_artifacts/libtorch_1742433629875/work/aten/src/ATen/cuda/CublasHandlePool.cpp:135.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
Processing prompts:   0%|          | 1/300 [00:01<09:34,  1.92s/it]

Computing gradient_x_input Scope...


Processing prompts:   1%|          | 2/300 [00:03<08:52,  1.79s/it]

...

Computing gradient_x_input Scope...


Processing prompts:  10%|█         | 30/300 [00:47<06:53,  1.53s/it]

Computing gradient_x_input Scope...


Processing prompts:  10%|█         | 31/300 [00:48<06:54,  1.54s/it]

Computing gradient_x_input Scope...


In [19]:
label

'Llama-3.2-3B__gradient_x_input_lambada_top0.05'

In [20]:
# Save results
result_path = Path(f"../results/{label}_results.json")
result_path.parent.mkdir(parents=True, exist_ok=True)
with open(result_path, "w", encoding="utf-8") as f:
    json.dump({"label": label, "results": results}, f, indent=2, ensure_ascii=False)

print(f"Saved {len(results)} results to {result_path}")

# Restore last successful result for visualization cells
if "_last_grad_vals" in dir():
    grad_vals, ablated_indices, tick_label_text = _last_grad_vals, _last_ablated, _last_tick
    logits_orig = _last_logits

Saved 300 results to ../results/Llama-3.2-3B__gradient_x_input_lambada_top0.05_results.json


In [21]:
## Summary of delta_log_prob for each prompt
for r in results:
    d = r.get("delta_log_prob")
    d_str = f"{d:.4f}" if d is not None else "ERROR"
    true_token = r.get('true_token', '')
    pred_token = r.get('predicted_token', '')
    is_correct = (true_token == pred_token)
    correctness = "✔" if is_correct else "✘"
    print(f"[{r['index']}] delta_log_prob={d_str}  true={true_token!r} pred={pred_token!r}  -> {correctness}")
    

[0] delta_log_prob=-6.8127  true=' couple' pred=' family'  -> ✘
[1] delta_log_prob=-9.1306  true=' snake' pred=' snake'  -> ✔
[2] delta_log_prob=-6.3876  true='ater' pred='ater'  -> ✔
[3] delta_log_prob=-1.3464  true=' wards' pred=' wards'  -> ✔
[4] delta_log_prob=-0.0058  true='f' pred='f'  -> ✔
[5] delta_log_prob=-3.4038  true=' truck' pred=' the'  -> ✘
[6] delta_log_prob=-6.7488  true=' stones' pred=' standing'  -> ✘
[7] delta_log_prob=-2.2038  true=' diamonds' pred=' bracelet'  -> ✘
[8] delta_log_prob=-15.2365  true='immer' pred='immer'  -> ✔
[9] delta_log_prob=-6.6949  true=' sled' pred=' sled'  -> ✔
[10] delta_log_prob=-0.2183  true=' fingers' pred=' fingers'  -> ✔
[11] delta_log_prob=-7.8777  true='raid' pred='raid'  -> ✔
[12] delta_log_prob=-5.6844  true=' pig' pred=' pig'  -> ✔
[13] delta_log_prob=-0.6739  true=' bundle' pred=' object'  -> ✘
[14] delta_log_prob=-2.2949  true=' quick' pred=' so'  -> ✘
[15] delta_log_prob=-0.6804  true=' terrible' pred=' indeed'  -> ✘
[16] delta

In [22]:
# Report correct prediction rate
num_correct = sum(1 for r in results if r.get('true_token', '') == r.get('predicted_token', ''))
total = len(results)
accuracy = num_correct / total if total > 0 else 0.0
print(f"\nCorrect prediction rate: {num_correct}/{total} = {accuracy:.2%}")

# Report average ablation impact (mean |delta_log_prob|) and its variance
deltas = [r["delta_log_prob"] for r in results if r.get("delta_log_prob") is not None]
avg_delta = sum(deltas) / len(deltas) if deltas else float("nan")
variance_delta = (
    sum((x - avg_delta) ** 2 for x in deltas) / len(deltas) if deltas else float("nan")
)
print(f"Average ablation impact (mean |delta_log_prob|): {avg_delta:.4f}")
print(f"Variance of ablation impact (|delta_log_prob|): {variance_delta:.6f}")
# Calculate standard error of the mean (SEM) for |delta_log_prob|
import math

if deltas and len(deltas) > 1:
    std_delta = math.sqrt(variance_delta)
    sem_delta = std_delta / math.sqrt(len(deltas))
else:
    sem_delta = float("nan")

print(f"Standard error of the mean (SEM) for |delta_log_prob|: {sem_delta:.6f}")

# Save variance, average, accuracy, and total to master_results.json (label as key)
master_path = Path("../results/master_results.json")
master_path.parent.mkdir(parents=True, exist_ok=True)
master = {}
if master_path.exists():
    with open(master_path, "r", encoding="utf-8") as f:
        master = json.load(f)
entry = {
    "avg_delta": avg_delta if deltas and not np.isnan(avg_delta) else None,
    "variance_delta": variance_delta if deltas and not np.isnan(variance_delta) else None,
    "sem_delta": sem_delta,
    "accuracy": accuracy,
    "num_correct": num_correct,
    "total": total,
    
}
master[label] = entry
with open(master_path, "w", encoding="utf-8") as f:
    json.dump(master, f, indent=2)
print(f"\nSaved to {master_path} (label={label})")



Correct prediction rate: 218/300 = 72.67%
Average ablation impact (mean |delta_log_prob|): -4.4897
Variance of ablation impact (|delta_log_prob|): 17.093762
Standard error of the mean (SEM) for |delta_log_prob|: 0.238703

Saved to ../results/master_results.json (label=Llama-3.2-3B__gradient_x_input_lambada_top0.05)
